In [ ]:
%load_ext autoreload
%autoreload 2

import humanoid_bench

from fast_td3.actors import ActorEGNN, Actor, ActorMPNN, ActorHEPI
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

In [ ]:
envs = HumanoidBenchEnv("h1-maze-v0", 4, device="cuda:0")

obs, xpos = envs.reset()

In [ ]:
actor = ActorEGNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=112,
    n_layers=2,
    init_scale=0.1,
    act_fn="relu"
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)

In [ ]:
actor = ActorMPNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=96,
    latent_dim=96,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)

In [ ]:
actor = ActorHEPI(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=64,
    latent_dim=64,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xpos)

In [ ]:
import torch
from torch_geometric.data import HeteroData
from collections import defaultdict


class H1:
    joint_dict = {
        "left_hip_yaw": 0,
        "left_hip_roll": 1,
        "left_hip_pitch": 2,
        "left_knee": 3,
        "left_ankle": 4,
        "right_hip_yaw": 5,
        "right_hip_roll": 6,
        "right_hip_pitch": 7,
        "right_knee": 8,
        "right_ankle": 9,
        "torso": 10,
        "left_shoulder_pitch": 11,
        "left_shoulder_roll": 12,
        "left_shoulder_yaw": 13,
        "left_elbow": 14,
        "right_shoulder_pitch": 15,
        "right_shoulder_roll": 16,
        "right_shoulder_yaw": 17,
        "right_elbow": 18,
    }

    edge_list = [
        (joint_dict["left_hip_yaw"], joint_dict["left_hip_roll"]),
        (joint_dict["left_hip_roll"], joint_dict["left_hip_pitch"]),
        (joint_dict["left_hip_pitch"], joint_dict["left_knee"]),
        (joint_dict["left_knee"], joint_dict["left_ankle"]),
        (joint_dict["right_hip_yaw"], joint_dict["right_hip_roll"]),
        (joint_dict["right_hip_roll"], joint_dict["right_hip_pitch"]),
        (joint_dict["right_hip_pitch"], joint_dict["right_knee"]),
        (joint_dict["right_knee"], joint_dict["right_ankle"]),
        (joint_dict["torso"], joint_dict["left_hip_yaw"]),
        (joint_dict["torso"], joint_dict["right_hip_yaw"]),
        (joint_dict["torso"], joint_dict["left_shoulder_pitch"]),
        (joint_dict["left_shoulder_pitch"], joint_dict["left_shoulder_roll"]),
        (joint_dict["left_shoulder_roll"], joint_dict["left_shoulder_yaw"]),
        (joint_dict["left_shoulder_yaw"], joint_dict["left_elbow"]),
        (joint_dict["torso"], joint_dict["right_shoulder_pitch"]),
        (joint_dict["right_shoulder_pitch"], joint_dict["right_shoulder_roll"]),
        (joint_dict["right_shoulder_roll"], joint_dict["right_shoulder_yaw"]),
        (joint_dict["right_shoulder_yaw"], joint_dict["right_elbow"]),
    ]

    edge_type_encoding = [
        0, 1, 2, 3,
        0, 1, 2, 3,
        4, 4,
        5, 6, 7, 8,
        5, 6, 7, 8,
    ]

    num_nodes = len(joint_dict)
    num_edges = len(edge_list)


# === Step 1: Joint type mapping ===
JOINT_TYPE_MAP = {
    "left_hip_yaw": "hip",
    "left_hip_roll": "hip",
    "left_hip_pitch": "hip",
    "right_hip_yaw": "hip",
    "right_hip_roll": "hip",
    "right_hip_pitch": "hip",
    "left_knee": "knee",
    "right_knee": "knee",
    "left_ankle": "ankle",
    "right_ankle": "ankle",
    "torso": "torso",
    "left_shoulder_pitch": "shoulder",
    "left_shoulder_roll": "shoulder",
    "left_shoulder_yaw": "shoulder",
    "left_elbow": "elbow",
    "right_shoulder_pitch": "shoulder",
    "right_shoulder_roll": "shoulder",
    "right_shoulder_yaw": "shoulder",
    "right_elbow": "elbow",
}


# === Step 2: Generate dummy features ===
angles = torch.randn(H1.num_nodes, 1)      # Replace with real angles
velocities = torch.randn(H1.num_nodes, 1)  # Replace with real velocities
node_features = torch.cat([angles, velocities], dim=1)  # Shape: [19, 2]


# === Step 3: Build HeteroData ===
data = HeteroData()
joint_names = list(H1.joint_dict.keys())
node_type_to_entries = defaultdict(list)  # {node_type: [(global_idx, joint_name)]}

# Group nodes by type
for joint_name, global_idx in H1.joint_dict.items():
    joint_type = JOINT_TYPE_MAP[joint_name]
    node_type_to_entries[joint_type].append((global_idx, joint_name))

# Add nodes by type
global_to_local = {}  # Maps: global_idx → (node_type, local_idx)
for node_type, entries in node_type_to_entries.items():
    indices = [idx for idx, _ in entries]
    feats = node_features[indices]  # [num_nodes_of_type, 2]
    data[node_type].x = feats
    for local_idx, (global_idx, _) in enumerate(entries):
        global_to_local[global_idx] = (node_type, local_idx)

# === Step 4: Add typed edges ===
for (src_global, dst_global), edge_type in zip(H1.edge_list, H1.edge_type_encoding):
    src_type, src_local = global_to_local[src_global]
    dst_type, dst_local = global_to_local[dst_global]
    edge_index = torch.tensor([[src_local], [dst_local]], dtype=torch.long)
    rel_name = f'edge_type_{edge_type}'
    data[(src_type, rel_name, dst_type)].edge_index = edge_index


# === Optional: Sanity check ===
print(data)
for ntype in data.node_types:
    print(f"Node type '{ntype}' has {data[ntype].x.shape[0]} nodes")

for etype in data.edge_types:
    print(f"Edge type {etype}: {data[etype].edge_index.shape[1]} edges")
